06/03/2026

Mik va a intentar hacer una red convolucional cn pytorch, lol

Estoy utilizando el env dl2024 

- Cloth**Dataset** guarda la info d vertices x caracteristicas () al acceder a estos items con el **DataLoader** le añade la otra dimension d frames(batchsize) para crear el tensor3D
- Redondear valores para optimizar (ahorra memoria)

Ahora el modelo itera con distintos batchSizes en modo shuffle, para ir entrenandose poco a poco. No tiene memoria, pero como guardamos las velocidades y tal probablemente funcione?
> Your model assumes that the current state of the cloth is all it needs to predict the next state (this is called a Markov assumption). In this setup, the network looks at a single frame's positions and velocities and predicts the displacements. Graph Neural Networks (GNNs) or standard Multi-Layer Perceptrons (MLPs) usually take data in this exact shape.

Otra idea sería:
> When to add a frame dimension (Sequence modeling): If your model needs temporal history—meaning it needs to look at, say, the last 5 frames to figure out what happens in the 6th frame. If you were using an LSTM, RNN, or a Spatiotemporal Transformer, your tensor would need to look like [Batch, Sequence_Length, Vertices, Features].
Por ahora no.

**Links Utilizados:**
- https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html
- https://lixiaoguang.medium.com/build-cnn-from-scratch-5-convolutional-neural-network-86b4d0323fb0

In [5]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import io
import torch

cloth_info = pd.read_csv('data/clothDataset_1_.csv')

print('cloth_info shape: {}'.format(cloth_info.shape))
print('cloth_info: \n{}'.format(cloth_info))

cloth_info shape: (30, 131)
cloth_info: 
    frame        x0        y0        z0        vx0        vy0       vz0  \
0       1 -0.002500 -0.005000  0.000000   0.000000   0.000000  0.000000   
1       2 -0.250000 -0.500000  0.000000 -12.375010 -24.750010  0.000000   
2       3 -0.249982 -0.499981  0.000000   0.000893   0.000939  0.000000   
3       4 -0.250010 -0.500045  0.000000  -0.001390  -0.003207  0.000000   
4       5 -0.250033 -0.499865  0.000000  -0.001153   0.009009  0.000000   
5       6 -0.249931 -0.499897  0.000000   0.005092  -0.001603  0.000000   
6       7 -0.250036 -0.500017  0.000000  -0.005226  -0.005975  0.000000   
7       8 -0.249945 -0.500060  0.000000   0.004528  -0.002176  0.000000   
8       9 -0.249992 -0.499946  0.000000  -0.002334   0.005707  0.000000   
9      10 -0.250081 -0.499962  0.000000  -0.004445  -0.000790  0.000000   
10     11 -0.250029 -0.499865  0.000000   0.002611   0.004849  0.000000   
11     12 -0.250076 -0.499875  0.000000  -0.002348  -0.0004

In [ ]:
class ClothDataset(Dataset):
    def __init__(self, csv_data, num_vertices=10):
        """
        Args:
            csv_data (str or filepath): Path to the CSV file or raw CSV string.
            num_vertices (int): Number of vertices per frame.
        """
        # Load the CSV data into a pandas DataFrame
        if isinstance(csv_data, str) and "frame,x0" in csv_data:
            self.data = pd.read_csv(io.StringIO(csv_data.strip()))
        else:
            self.data = pd.read_csv(csv_data)
            
        #quick fix para espacios en primera fila
        self.data.columns = self.data.columns.str.strip()
        
        self.num_vertices = num_vertices
        
        # Define the base feature names to extract per vertex
        self.feature_prefixes = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'sdf', 'nx', 'ny', 'nz']

    def __len__(self):
        # The number of items is the number of frames (rows) in the dataset
        return len(self.data) - 1  # El ultimo frame NO tiene siguiente frame

    def _get_frame_tensor(self, idx):
        row = self.data.iloc[idx]
        frame_data = []

        for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.feature_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

        tensor_data = torch.tensor(np.array(frame_data))

        return tensor_data
        
    def __getitem__(self, idx):
        frame_t = self._get_frame_tensor(idx)
        frame_t1 = self._get_frame_tensor(idx + 1) # TO DO: Pillar solo las columnas de pos

        return frame_t, frame_t1

# --- Example Usage ---

# (Assuming 'csv_string' is a variable holding your provided data block)
dataset = ClothDataset('data/clothDataset_1_.csv')
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

for batch_data, batch_frames in dataloader:
    print(f"Batch Shape: {batch_data.shape}") # Expected: [4, 10, 13]
    print(batch_data)
    print(batch_frames)
    break

Batch Shape: torch.Size([4, 10, 10])
tensor([[[-2.5002e-01, -4.9998e-01,  0.0000e+00, -3.0316e-03,  2.1786e-03,
           0.0000e+00, -5.0013e-01, -2.5000e-01,  0.0000e+00, -4.5270e-03],
         [-2.1428e-03,  0.0000e+00, -5.0008e-01, -5.0005e-01,  0.0000e+00,
          -4.4197e-03, -5.3465e-03,  0.0000e+00, -2.5007e-01, -2.5002e-01],
         [ 0.0000e+00,  5.6475e-04, -2.8312e-04,  0.0000e+00, -1.0039e-04,
          -4.9997e-01,  0.0000e+00, -3.4733e-03, -5.8711e-04,  0.0000e+00],
         [-5.0003e-01, -4.9500e-05,  0.0000e+00,  7.6592e-04, -4.0312e-03,
           0.0000e+00, -6.4178e-05, -2.4996e-01,  0.0000e+00, -5.1414e-03],
         [ 2.8141e-03,  0.0000e+00,  2.4996e-01, -5.0002e-01,  0.0000e+00,
          -3.3952e-03, -7.8782e-03,  0.0000e+00, -2.5004e-01,  3.3799e-05],
         [ 0.0000e+00, -1.1072e-03, -4.9604e-04,  0.0000e+00, -4.9994e-01,
           2.4998e-01,  0.0000e+00,  1.4171e-03,  1.8872e-03,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e

In [ ]:
# TODO
# una recurrente sencilla (la salida se vuelve entrada en el siguiente ejemplo)
# Antes de meternos en CNN y LSTM
# AÑADIMOS VALORES U V PARA CADA VERTICE ( no queremos perder la noción espacial )

import torch.nn as nn
import torch.nn.functional as F #acceso rapido a funciones
import torch.utils.data as data #cargar y manejar el training data

class MyModule(nn.Module):

    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()
        # Some init for my module
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.relu = nn.ReLu()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Function for performing the calculation of the module.
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        return x

    #backward se hace automaticamente, podriamos definirla tmbn 

#tmbn clases DataSet y DataLoader

# definir modelo, loss function y optimizer
#TODO: buscar dimensiones reales de las neuronas
model = MyModule(num_inputs=dataloader.shape[1], num_hidden=128, num_outputs=1)
# print, save, lo que sea
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    for batch_t, batch_t1 in dataloader:
        optimizer.zero_grad()
        pred = model(batch_t)
        loss = criterion(pred, batch_t1)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')


In [8]:
import torch
import torch.nn as nn

#EJEMPLO SENCILLO CONVOLUCIONAL PARA MÁS ADELANTE

# Example: 100 features, 1 channel (linear input)
# Batch size=16
input_data = torch.randn(16, 1, 100) 

model = nn.Sequential(
    nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3), # Extract features
    nn.ReLU(),
    nn.Flatten(), # Flatten for Dense layer
    nn.Linear(32 * 98, 10) # 98 is the new length after convolution
)
